In [ ]:
from arize import ArizeClient
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")

import os, certifi
CA = certifi.where()

os.environ["SSL_CERT_FILE"] = CA
os.environ["REQUESTS_CA_BUNDLE"] = CA
os.environ["CURL_CA_BUNDLE"] = CA

print("Using CA:", CA)


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("cais/mmlu", "high_school_psychology")

In [ ]:
df = ds["test"].to_pandas()
df.head()

In [ ]:
client = ArizeClient(api_key=API_KEY)

In [ ]:
import numpy as np

for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, np.ndarray)).any():
        df[col] = df[col].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)


In [ ]:
dataset = client.datasets.create(name="mmlu_high_school_psychology", space=SPACE_ID, examples=df)

In [ ]:
# Define your task
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    # model="gpt-5.1",
    temperature=0,  # 생성적 응답의 다양성
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

prompt = ChatPromptTemplate.from_template("""
You are an expert in answering questions.

Find the correct answer to the question from the choices.

Your response must contain ONLY the index number of the correct answer. Index starts from 0.

Do not include any explanatory text, labels, phrases like "The correct answer is:", or additional words. Output only the single digit.

Question: {input}

Choices: {choices}
"""
)

chain = prompt | llm | StrOutputParser()

def sample_task(dataset_row) -> str:
    response = chain.invoke({"input": dataset_row["question"], "choices": dataset_row["choices"]})
    return response

In [ ]:
sample_task(df.iloc[1])

In [ ]:
from arize.experiments import EvaluationResult

def is_correct(output, dataset_row):
    ground_truth = int(dataset_row.get("answer"))
    correct = ground_truth == int(output)
    return EvaluationResult(
        score=int(correct),
        label="correct" if correct else "incorrect",
        explanation="Evaluator explanation here"
    )

In [ ]:
experiment, experiment_df = client.experiments.run(
    name="MMLU-Experiment-gpt-4o-mini",
    dataset=dataset.id,
    task=sample_task,
    evaluators=[is_correct],
)